# Apply Rating Model

In [ ]:
"""
Apply the trained player rating model to transformed player statistics.

This script enriches player statistics with position and match context, predicts
ratings with rating_model.pkl, and writes the updated player_stats.csv file.
"""

from pathlib import Path
import sys
from typing import Iterable

import pandas as pd

sys.path.append(str(Path.cwd().parent))

from toolkit.ml_utilities import load_model


PLAYER_STATS_PATH = Path("../data/transform/player_stats.csv")
PLAYERS_PATH = Path("../data/transform/players.csv")
MATCHES_PATH = Path("../data/transform/matches.csv")
MODEL_PATH = Path("rating_model.pkl")

OUTPUT_PATH = PLAYER_STATS_PATH

NUMERIC_COLUMNS = [
    "goals",
    "assists",
    "minutes",
    "on_min",
    "off_min",
    "team_goals",
    "team_conceded",
]

CATEGORICAL_COLUMNS = [
    "position",
    "result",
]

BOOLEAN_COLUMNS = [
    "yellow",
    "yellow_red",
    "red",
    "start_eleven",
]

FEATURE_COLUMNS = NUMERIC_COLUMNS + BOOLEAN_COLUMNS + CATEGORICAL_COLUMNS

PLAYER_COLUMNS = [
    "player_id",
    "position",
]

MATCH_COLUMNS = [
    "match_id",
    "date",
    "home_club_id",
    "away_club_id",
    "home_goals",
    "away_goals",
]


def validate_columns(dataframe: pd.DataFrame, required_columns: Iterable[str]) -> None:
    """Raise an error when required columns are missing from a dataframe."""
    missing_columns = sorted(set(required_columns) - set(dataframe.columns))

    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")


def load_csv(input_path: Path, required_columns: Iterable[str]) -> pd.DataFrame:
    """Load a CSV file and validate its required columns."""
    if not input_path.exists():
        raise FileNotFoundError(f"File not found: {input_path}")

    dataframe = pd.read_csv(input_path)
    validate_columns(dataframe, required_columns)
    return dataframe


def add_player_positions(
    player_stats: pd.DataFrame,
    players: pd.DataFrame,
) -> pd.DataFrame:
    """Add player positions to player statistics."""
    if "position" in player_stats.columns:
        return player_stats.copy()

    return player_stats.merge(
        players[PLAYER_COLUMNS],
        on="player_id",
        how="left",
    )


def add_match_context(
    player_stats: pd.DataFrame,
    matches: pd.DataFrame,
) -> pd.DataFrame:
    """Add match context columns to player statistics."""
    return player_stats.merge(
        matches[MATCH_COLUMNS],
        on="match_id",
        how="left",
    )


def add_match_result(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Add the match result from the perspective of the player's club."""
    result = dataframe.copy()

    is_home_team = result["club_id"] == result["home_club_id"]
    is_away_team = result["club_id"] == result["away_club_id"]

    result["result"] = None

    result.loc[
        is_home_team & (result["home_goals"] > result["away_goals"]),
        "result",
    ] = "win"
    result.loc[
        is_home_team & (result["home_goals"] < result["away_goals"]),
        "result",
    ] = "loss"
    result.loc[
        is_home_team & (result["home_goals"] == result["away_goals"]),
        "result",
    ] = "draw"

    result.loc[
        is_away_team & (result["away_goals"] > result["home_goals"]),
        "result",
    ] = "win"
    result.loc[
        is_away_team & (result["away_goals"] < result["home_goals"]),
        "result",
    ] = "loss"
    result.loc[
        is_away_team & (result["away_goals"] == result["home_goals"]),
        "result",
    ] = "draw"

    return result


def predict_ratings(
    dataframe: pd.DataFrame,
    model,
) -> pd.DataFrame:
    """Predict and append player ratings."""
    validate_columns(dataframe, FEATURE_COLUMNS)

    result = dataframe.copy()
    result["rating"] = model.predict(result[FEATURE_COLUMNS]).round(1)
    return result


def select_output_columns(
    dataframe: pd.DataFrame,
    original_columns: list[str],
) -> pd.DataFrame:
    """Keep original player statistics columns and the predicted rating column."""
    output_columns = list(original_columns)

    if "rating" not in output_columns:
        output_columns.append("rating")

    return dataframe[output_columns]


def main() -> None:
    """Run rating prediction and update the player statistics CSV file."""
    player_stats = load_csv(
        PLAYER_STATS_PATH,
        required_columns=["player_id", "match_id", "club_id"],
    )
    players = load_csv(PLAYERS_PATH, required_columns=PLAYER_COLUMNS)
    matches = load_csv(MATCHES_PATH, required_columns=MATCH_COLUMNS)

    original_columns = list(player_stats.columns)

    enriched_stats = add_player_positions(player_stats, players)
    enriched_stats = add_match_context(enriched_stats, matches)
    enriched_stats = add_match_result(enriched_stats)

    model = load_model(str(MODEL_PATH))
    predicted_stats = predict_ratings(enriched_stats, model)
    output_stats = select_output_columns(predicted_stats, original_columns)

    output_stats.to_csv(OUTPUT_PATH, index=False)
    print(f"[INFO] Ratings written to: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()
